In [ ]:
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt


from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, auc, accuracy_score, roc_auc_score,f1_score,log_loss,\
classification_report, roc_curve

import warnings
warnings.filterwarnings("ignore");

RAND = 10

# Import danych


**Opis pól**

- client_id — identyfikator klienta (numer porządkowy)
- education — wykształcenie
- sex — płeć
- age — wiek
- car — posiadanie samochodu
- car_type — model samochodu
- decline_app_cnt — liczba odrzuconych wniosków
- good_work — wskaźnik „dobrej” pracy / zatrudnienia
- bki_request_cnt — liczba zapytań do BKI
- home_address — kategoryzacja adresu zamieszkania
- work_address — kategoryzacja adresu pracy
- income — dochód kredytobiorcy
- foreign_passport — paszport (podróże zagraniczne)
- sna — relacja kredytobiorcy z pracownikami banku
- first_time — pierwsza wzmianka o kliencie (czas)
- score_bki — scoring według danych BKI
- region_rating — rating regionu
- app_date — data złożenia wniosku
- default — niespłata kredytu (zmienna celu)


In [ ]:
df = pd.read_csv('../data/train.csv')


In [ ]:
df

In [ ]:
print(f'Rozmiar zbioru = {df.shape}')


In [ ]:
df.info()

In [ ]:
df.education.isna().sum() / df.shape[0]*100

In [ ]:
df.education.unique()

- SCH — szkoła / wykształcenie

- GRD — absolwent studiów (*graduate*)

- Student / licencjat — UGR (*undergraduate*)

- PGR — studia podyplomowe (*postgraduate*)

- ACD — środowisko akademickie (*academician*)


In [ ]:
ed_mode = df.education.mode()[0]
df.education = df.education.fillna(ed_mode)

Przekształcamy wybrane cechy na pola kategoryczne; kategorie odpowiadają wartościom warunkowym w danych.


In [ ]:
df.describe()

In [ ]:
df.nunique()

In [ ]:
df.home_address.unique()

In [ ]:
df[['home_address', 'work_address']] = df[['home_address', 'work_address']].astype(object)

In [ ]:
df.describe(include=object)

# Eksploracyjna analiza danych (EDA)


**Hipotezy:**

- Wiek „dobrych” kredytobiorców jest wyższy niż „złych” (rozkład wieku zależy od default i jest przesunięty przy default=0)
- Poziom wykształcenia zależy od wieku, co wpływa na spłatę; osoby z wyższym wykształceniem częściej są „dobrymi” klientami
- Przy `good_work = 0` rośnie ryzyko niespłaty (flaga default)
- Dochód „dobrych” jest wyższy niż „złych” (rozkład dochodu zależy od default)
- `score_bki` jest powiązany z default: im mniejszy, tym wyższe ryzyko problemów ze spłatą


## Zmienna celu


Najpierw warto przeanalizować rozkład zmiennej celu (`default`).


In [ ]:
# Udział klas (skala procentowa)
norm_target = (df
               .default
               .value_counts(normalize=True)
               .mul(100)
               .rename('percent')
               .reset_index())

plt.figure(figsize=(15, 7))
ax = sns.barplot(x='index', y='percent', data=norm_target)

for p in ax.patches:
    percentage = '{:.1f}%'.format(p.get_height())
    ax.annotate(percentage,
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center',
                va='center',
                xytext=(0, 10),
                textcoords='offset points',
                fontsize=14)

plt.title('Rozkład default', fontsize=20)

plt.xlabel('default', fontsize=14)
plt.ylabel('Odsetek (%)', fontsize=14)

plt.xticks(fontsize=14)
plt.yticks(fontsize=14);


W zbiorze jest nierównowaga klas — należy ją uwzględnić przy uczeniu modelu.


## Age

In [ ]:
sns.displot(
    {
        "brak niespłaty (0)": df[df.default == 0].age,
        "niespłata (1)": df[df.default == 1].age,
    },
    kind="kde",
    common_norm=False
)

plt.title('Wiek', fontsize=20)
plt.xlabel('Wiek', fontsize=14)
plt.ylabel('Gęstość', fontsize=14)

plt.xticks(fontsize=14)
plt.yticks(fontsize=14);


In [ ]:
df.groupby('default')['age'].median()

In [ ]:
df.groupby('default')['age'].mean()

In [ ]:
df.groupby('default')['age'].apply(lambda x: x.value_counts().index[0]).reset_index()

Są niewielkie różnice wieku w podziale na default; hipoteza została w przybliżeniu potwierdzona.


## Wykształcenie


In [ ]:
sns.displot(
    {
        "wykształcenie SCH": df[df.education == 'SCH'].age,
        "wykształcenie GRD": df[df.education == 'GRD'].age,
        "wykształcenie UGR": df[df.education == 'UGR'].age,
        "wykształcenie PGR": df[df.education == 'PGR'].age,
        "wykształcenie ACD": df[df.education == 'ACD'].age,
    },
    kind="kde",
    common_norm=False
)

plt.title('Wiek a wykształcenie', fontsize=20)
plt.xlabel('Wiek', fontsize=14)
plt.ylabel('Gęstość', fontsize=14)

plt.xticks(fontsize=14)
plt.yticks(fontsize=14);


Przyjmując SCH (być może inna interpretacja skrótu), rozkład wieku jest przesunięty w górę względem UGR. ACD charakteryzuje się przesunięciem rozkładu wieku w lewo w porównaniu do PGR — co jest intuicyjne. Najmłodsi to UGR; istnieje przypuszczenie, że częściej będą „gorszymi” kredytobiorcami.


In [ ]:
plt.figure(figsize=(15, 7))

sns.boxplot(x='education', y='age', data=df)

plt.title('Wykres pudełkowy: wiek i wykształcenie', fontsize=20)
plt.ylabel('Wiek', fontsize=14)
plt.xlabel('Wykształcenie', fontsize=14)

plt.xticks(fontsize=14)
plt.yticks(fontsize=14);


In [ ]:
plt.figure(figsize=(15, 7))

sns.boxplot(x='education', y='age', hue='default', data=df)

plt.title('Wiek, wykształcenie i default', fontsize=20)
plt.ylabel('Wiek', fontsize=14)
plt.xlabel('Wykształcenie', fontsize=14)

plt.xticks(fontsize=14)
plt.yticks(fontsize=14);


Ciekawe jest to, że dla PGR i ACD średni wiek „złych” kredytobiorców jest wyższy niż „dobrych”, przy dużej rozpiętości wartości. Być może w przedziale np. 30–50 lat wpływ ten na pełną spłatę będzie mniejszy.


In [ ]:
df.groupby('education')['age'].median()

In [ ]:
plt.figure(figsize=(15, 7))

educ_def = (df.groupby(['default'])['education']
            .value_counts(normalize=True)
            .rename('percentage')
            .mul(100)
            .reset_index()
            .sort_values('education'))

ax = sns.barplot(x="education", y="percentage", hue="default", data=educ_def)

for p in ax.patches:
    percentage = '{:.1f}%'.format(p.get_height())
    ax.annotate(percentage,
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center',
                va='center',
                xytext=(0, 10),
                textcoords='offset points',
                fontsize=14)

plt.title('Wykształcenie a default', fontsize=20)
plt.ylabel('Odsetek (%)', fontsize=14)
plt.xlabel('default', fontsize=14)

plt.xticks(fontsize=14)
plt.yticks(fontsize=14);


Podobnie widać, że odsetek nierzetelnych jest wyższy przy poziomie SCH; dla UGR różnica nie jest duża. To raczej nie sam wiek, co wykształcenie ma wpływ. Częściowo można uznać, że absolwenci (GRD) są stabilniejsi w spłatach.


## Korelacje


In [ ]:
num_cols = ['age', 'decline_app_cnt', 'good_work', 'bki_request_cnt',
       'region_rating', 'income', 'sna', 'first_time', 'score_bki']

plt.figure(figsize=(10, 8))

sns.heatmap(df[num_cols].corr(method='spearman'), annot=True, fmt=".1f");


## Dochód


In [ ]:
sns.displot(df, x='income', hue='default', kind="kde", common_norm=False)

plt.xlabel('Dochód', fontsize=14)
plt.ylabel('Gęstość', fontsize=14)

plt.xticks(fontsize=14)
plt.yticks(fontsize=14);


In [ ]:
plt.figure(figsize=(15, 6))

sns.boxplot(x='default', y='income', data=df)

plt.ylabel('Dochód', fontsize=14)
plt.xlabel('Default', fontsize=14)

plt.xticks(fontsize=14)
plt.yticks(fontsize=14);


In [ ]:
df.groupby('default')['income'].mean()

In [ ]:
df.groupby('default')['income'].median()

In [ ]:
df_income = df.copy()
df_income.income = np.log(df.income+1)

In [ ]:
sns.displot(
    {
        "wykształcenie SCH": df_income[df_income.education == 'SCH'].income,
        "wykształcenie GRD": df_income[df_income.education == 'GRD'].income,
        "wykształcenie UGR": df_income[df_income.education == 'UGR'].income,
        "wykształcenie PGR": df_income[df_income.education == 'PGR'].income,
        "wykształcenie ACD": df_income[df_income.education == 'ACD'].income,
    },
    kind="kde",
    common_norm=False
)

plt.title('Dochód a wykształcenie', fontsize=20)
plt.xlabel('Dochód', fontsize=14)
plt.ylabel('Gęstość', fontsize=14)

plt.xticks(fontsize=14)
plt.yticks(fontsize=14);


…


# Inżynieria cech


In [ ]:
df.columns

In [ ]:
num_cols = ['age', 'decline_app_cnt', 'score_bki',
            'bki_request_cnt', 'income', 'first_time','region_rating']

In [ ]:
numeric_features = df[num_cols]

numeric_features = numeric_features.stack().reset_index().rename(
    columns={"level_1": "Features", 0: "Value"})

ax = sns.FacetGrid(data=numeric_features, col="Features",
                  col_wrap=3, sharex=False, sharey=False)
ax = ax.map(sns.distplot, "Value")

plt.subplots_adjust(top=0.9)
plt.suptitle("Histogramy wybranych cech");


In [ ]:
for i in ['age', 'decline_app_cnt', 'bki_request_cnt', 'income']:
    df[i] = np.log(df[i]+1)

In [ ]:
numeric_features = df[num_cols]

numeric_features = numeric_features.stack().reset_index().rename(
    columns={"level_1": "Features", 0: "Value"})

ax = sns.FacetGrid(data=numeric_features, col="Features",
                  col_wrap=3, sharex=False, sharey=False)
ax = ax.map(sns.distplot, "Value")

plt.subplots_adjust(top=0.9)
plt.suptitle("Histogramy wybranych cech (po ln(1+x))");


In [ ]:
# Nowa cecha — miesiąc
df['app_date'] = pd.to_datetime(df['app_date'], format='%d%b%Y')
df['month'] = df['app_date'].dt.month.astype(object)

df.drop(['app_date'],  axis=1, inplace=True)


In [ ]:
# Średni dochód z uwzględnieniem ratingu regionu

mean_inc_reg = df.groupby('region_rating')['income'].median().to_dict()
df['mean_income_region'] = df['region_rating'].map(mean_inc_reg)


In [ ]:
# Średni dochód z uwzględnieniem wieku (po ln)

mean_inc_age = df.groupby('age')['income'].median().to_dict()
df['mean_income_age'] = df['age'].map(mean_inc_age)


In [ ]:
# Średni score_bki z uwzględnieniem wieku

mean_bki_age = df.groupby('age')['score_bki'].median().to_dict()
df['mean_bki_age'] = df['age'].map(mean_bki_age)


In [ ]:
df[:5]

In [ ]:
cat_cols = ['education', 'sex', 'car', 'car_type', 'good_work', 'home_address', 'work_address',
       'foreign_passport', 'sna', 'month']

In [ ]:
df.info()

In [ ]:
num_cols = list(df.dtypes[df.dtypes == float].index) + list(df.dtypes[df.dtypes == int].index)[1:-1]

In [ ]:
num_cols

# Modelowanie


In [ ]:
df_label = pd.get_dummies(df, columns=cat_cols,
                          drop_first=True).drop('client_id', axis=1)

In [ ]:
df_label[:5]


In [ ]:
X = df_label.drop('default', axis=1)
y = df_label['default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, shuffle=True, random_state=RAND)

## Baseline


In [ ]:
lr = LogisticRegression(class_weight = 'balanced')
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)
y_score = lr.predict_proba(X_test)[:,1]

In [ ]:
print('ROC-AUC:', roc_auc_score(y_test, y_score))
print('Precyzja:', precision_score(y_test, y_pred))
print('Czułość (recall):', recall_score(y_test, y_pred))
print('F1:', f1_score(y_test, y_pred))
print('Log-loss:', log_loss(y_test, y_pred))


Jeśli bankowi bardziej opłaca się wychwycić jak najwięcej „złych” klientów (uniknąć dużych strat), warto optymalizować **recall (czułość)** przy porównywaniu baseline i innych modeli; w przeciwnym razie częściej stosuje się **precision (precyzję)**.


![precision](http://3.bp.blogspot.com/-cS83mPIWFqU/Wuq1vw0TrpI/AAAAAAAACY8/qEVMoTUmvkIwNbtbrELeGhJoC4yYsrVrQCK4BGAYYCw/s1600/prec.png)

![recall](http://1.bp.blogspot.com/-_-bHkzJAneg/Wuq1oUu_vTI/AAAAAAAACY0/E2UIsd359x0GV_9IgGv0Iax6Z3kGYM9dQCK4BGAYYCw/s1600/rec.png)


Dodawanie metryk do tabeli


In [ ]:
metrics = pd.DataFrame(
    index=['ROC-AUC', 'precyzja', 'czulość', 'F1', 'log-loss'])

metrics['Baseline_LR'] = [roc_auc_score(y_test, y_score),
                          precision_score(y_test, y_pred),
                          recall_score(y_test, y_pred),
                          f1_score(y_test, y_pred),
                          log_loss(y_test, y_pred)]


In [ ]:
metrics

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_score)

roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, color='darkorange', label='Krzywa ROC (pole = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])

plt.xlabel('Odsetek fałszywie pozytywnych (FPR)')
plt.ylabel('Odsetek prawdziwie pozytywnych (TPR)')

plt.title('Regresja logistyczna — ROC AUC = %0.3f' % roc_auc)
plt.legend(loc="lower right")
plt.show()


# Dobór hiperparametrów


In [ ]:
parameters_grid = {
    'penalty': ['l1', 'l2', 'elasticnet'],
    'C': np.linspace(1, 1000, num=5),
    'solver': ['sag', 'saga', 'lbfgs'],
    'l1_ratio': [0.25, 0.5, 0.75],
    'max_iter': np.arange(100, 500, 100)
}

lr = LogisticRegression(class_weight='balanced')
cv = StratifiedKFold(n_splits=3, shuffle=True)
grid_cv = GridSearchCV(lr, parameters_grid,
                       scoring='roc_auc', cv=cv, verbose=2)

In [ ]:
%%time
# Trening z przeszukiwaniem siatki (odkomentuj, aby włączyć)
# grid_cv.fit(X_train, y_train)


In [ ]:
# print(grid_cv.best_score_)
# print(grid_cv.best_params_)


In [ ]:
best_params = {'C': 500.5, 
               'l1_ratio': 0.25,
               'max_iter': 400, 
               'penalty': 'l2', 
               'solver': 'lbfgs'}

In [ ]:
lr_gr = LogisticRegression(**best_params, class_weight='balanced')
lr_gr.fit(X_train, y_train)

y_pred_gr = lr_gr.predict(X_test)
y_score_gr = lr_gr.predict_proba(X_test)[:,1]

In [ ]:
print('ROC-AUC:', roc_auc_score(y_test, y_score_gr))
print('Precyzja:', precision_score(y_test, y_pred_gr))
print('Czułość (recall):', recall_score(y_test, y_pred_gr))
print('F1:', f1_score(y_test, y_pred_gr))
print('Log-loss:', log_loss(y_test, y_pred_gr))


In [ ]:
fpr_2, tpr_2, thresholds_2 = roc_curve(y_test, y_score_gr)

roc_auc_2 = auc(fpr_2, tpr_2)

plt.plot(fpr, tpr, color='darkorange', label='Krzywa ROC (pole = %0.2f)' % roc_auc)
plt.plot(fpr_2, tpr_2, color='green', label='Krzywa ROC (pole = %0.2f)' % roc_auc_2)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])

plt.xlabel('Odsetek fałszywie pozytywnych (FPR)')
plt.ylabel('Odsetek prawdziwie pozytywnych (TPR)')

plt.title('Regresja logistyczna — porównanie ROC (baseline vs siatka)')
plt.legend(loc="lower right")
plt.show()


In [ ]:
metrics['Grid_LR'] = [roc_auc_score(y_test, y_score_gr),
                      precision_score(y_test, y_pred_gr),
                      recall_score(y_test, y_pred_gr),
                      f1_score(y_test, y_pred_gr),
                      log_loss(y_test, y_pred_gr)]

In [ ]:
metrics[:-1].style.highlight_max(axis=1, color='lightblue')

# Analiza ważnych cech (SHAP)


In [ ]:
import shap

In [ ]:
explainer = shap.LinearExplainer(lr_gr, X_train, feature_dependence="independent")
shap_values = explainer(X_test)

In [ ]:
# Wykres podsumowujący SHAP
shap.summary_plot(shap_values, X_test)


Cechy są uporządkowane wg ważności wzdłuż osi Y; wzdłuż OX — wartości Shapleya. Każdy punkt to pojedyncza obserwacja.

Kolor oznacza wartość atrybutu: czerwony — wysoka, niebieski — niska.

**Przykład interpretacji**

- im wyższa wartość `score_bki`, tym wyższe prawdopodobieństwo defaultu;
- im wyższy wiek, tym niższe prawdopodobieństwo defaultu (w tej interpretacji SHAP).


Porównanie z wagami regresji logistycznej oraz kilkoma wybranymi cechami.


In [ ]:
lr_gr.coef_

In [ ]:
feature_imp = pd.DataFrame()
feature_imp['feature'] = X.columns
feature_imp['imp'] = lr_gr.coef_[0]
feature_imp = feature_imp.sort_values(by='imp', ascending=False)

In [ ]:
feature_imp

In [ ]:
df_label.groupby('default')[['score_bki']].median()

In [ ]:
vals = np.abs(shap_values.values).mean(0)
feature_names = X_train

feature_importance = pd.DataFrame(list(zip(feature_names, vals)),
                                  columns=['col_name', 'feature_importance_vals'])
feature_importance.sort_values(by=['feature_importance_vals'],
                               ascending=False, inplace=True)

In [ ]:
cols_show = list(set(feature_importance.col_name[:15]) & set(num_cols))

In [ ]:
cols_show

In [ ]:
df_label[cols_show + ['default']].groupby('default')[cols_show].mean().T